# 🌳 Tree-of-Thought 与图推理

今天我们要学习一种更强大的推理方法：**Tree-of-Thought**（思维树），它比传统的Chain-of-Thought更加灵活和强大！

## 🎯 学习目标
- 理解Tree-of-Thought的核心思想
- 掌握图推理的基本概念
- 学习分支探索和剪枝策略
- 能够实现简单的思维树推理

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from typing import List, Dict, Tuple
import json

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 🔍 什么是Tree-of-Thought？

**Tree-of-Thought**是一种让模型同时探索多条推理路径的方法，就像我们解决问题时会有多种思路一样。

### 📚 核心概念
- **分支**：每条独立的推理路径
- **节点**：推理过程中的中间结论
- **边**：推理步骤之间的关系
- **剪枝**：淘汰不合理的分支
- **自洽性**：确保各分支逻辑一致

🌟 **优势**：相比CoT的线性推理，ToT可以并行探索，找到最优解！

In [ ]:
def draw_tree_of_thought():
    """绘制思维树示意图"""
    G = nx.DiGraph()
    
    # 添加节点
    nodes = ['问题', '方法A', '方法B', '方法C', '答案A', '答案B', '答案C', '最优解']
    G.add_nodes_from(nodes)
    
    # 添加边
    edges = [('问题', '方法A'), ('问题', '方法B'), ('问题', '方法C'),
             ('方法A', '答案A'), ('方法B', '答案B'), ('方法C', '答案C'),
             ('答案A', '最优解'), ('答案B', '最优解'), ('答案C', '最优解')]
    G.add_edges_from(edges)
    
    # 绘制图形
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(12, 8))
    
    nx.draw(G, pos, with_labels=True, node_color='lightblue', 
            node_size=2000, font_size=10, font_weight='bold',
            arrows=True, arrowsize=20, edge_color='gray')
    
    plt.title('Tree-of-Thought 思维树结构', fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

draw_tree_of_thought()

## 🧮 实践示例：计算平方根

让我们用Tree-of-Thought来解决一个问题：**25的平方根是多少？**

### 🌲 思维树结构
```
问题: √25 = ?
├── 分支A: 试错法
│   ├── 5×5=25 → 答案: 5
└── 分支B: 因数分解
    ├── 25=5×5 → √25=5
    └── 验证: 5×5=25 ✅
```

现在让我们用代码实现这个逻辑！

In [ ]:
class TreeOfThought:
    """思维树推理类"""
    
    def __init__(self, problem: str):
        self.problem = problem
        self.branches = []
        self.final_answer = None
    
    def add_branch(self, method: str, reasoning: str, answer):
        """添加一个推理分支"""
        branch = {
            'method': method,
            'reasoning': reasoning,
            'answer': answer,
            'is_consistent': True
        }
        self.branches.append(branch)
    
    def check_consistency(self):
        """检查各分支的自洽性"""
        if len(self.branches) < 2:
            return
        
        # 如果所有分支都得到相同答案，则自洽
        answers = [branch['answer'] for branch in self.branches]
        unique_answers = set(answers)
        
        for branch in self.branches:
            if branch['answer'] not in unique_answers:
                branch['is_consistent'] = False
    
    def find_best_answer(self):
        """找到最佳答案"""
        consistent_branches = [b for b in self.branches if b['is_consistent']]
        
        if consistent_branches:
            # 自洽的分支优先
            self.final_answer = consistent_branches[0]['answer']
        else:
            # 如果没有自洽分支，选择出现频率最高的答案
            answers = [b['answer'] for b in self.branches]
            answer_counts = {}
            for ans in answers:
                answer_counts[ans] = answer_counts.get(ans, 0) + 1
            self.final_answer = max(answer_counts.items(), key=lambda x: x[1])[0]
    
    def display(self):
        """显示思维树结构"""
        print(f"🤔 问题: {self.problem}")
        print("\n🌲 推理分支:")
        
        for i, branch in enumerate(self.branches, 1):
            status = "✅" if branch['is_consistent'] else "❌"
            print(f"\n分支{i} - {branch['method']} {status}")
            print(f"  推理过程: {branch['reasoning']}")
            print(f"  得出答案: {branch['answer']}")
        
        if self.final_answer:
            print(f"\n🎯 最终答案: {self.final_answer}")
        
        return self.final_answer

# 使用思维树解决平方根问题
tot = TreeOfThought("25的平方根是多少？")

# 分支1: 试错法
tot.add_branch(
    method="试错法",
    reasoning="尝试不同的数字，直到找到5×5=25",
    answer=5
)

# 分支2: 因数分解
tot.add_branch(
    method="因数分解",
    reasoning="25=5×5，所以√25=5",
    answer=5
)

# 分支3: 开方法
tot.add_branch(
    method="开方法",
    reasoning="√25 = √(5²) = 5",
    answer=5
)

# 检查自洽性并找到最佳答案
tot.check_consistency()
tot.find_best_answer()

tot.display()

## 🔄 图推理的实现

图推理将问题建模为图结构，通过寻找最优路径来解决复杂问题。让我们实现一个简单的图推理示例！

In [ ]:
def draw_graph_reasoning():
    """绘制图推理示例"""
    G = nx.Graph()
    
    # 添加节点
    nodes = ['起点A', '节点B', '节点C', '节点D', '终点E']
    G.add_nodes_from(nodes)
    
    # 添加边（带权重）
    edges = [('起点A', '节点B', 3), ('起点A', '节点C', 1),
             ('节点B', '节点D', 2), ('节点C', '节点D', 4),
             ('节点D', '终点E', 2)]
    
    for edge in edges:
        G.add_edge(edge[0], edge[1], weight=edge[2])
    
    # 绘制图形
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(10, 6))
    
    # 绘制边
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
    
    # 绘制节点
    nx.draw(G, pos, with_labels=True, node_color='lightgreen', 
            node_size=1500, font_size=10, font_weight='bold',
            edge_color='gray', width=2)
    
    plt.title('图推理：最短路径问题', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

draw_graph_reasoning()

In [ ]:
def find_shortest_path(graph, start, end):
    """使用Dijkstra算法寻找最短路径"""
    import heapq
    
    # 初始化距离字典
    distances = {node: float('infinity') for node in graph.nodes()}
    distances[start] = 0
    
    # 优先队列
    priority_queue = [(0, start)]
    
    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)
        
        # 如果到达终点
        if current_node == end:
            return current_distance
        
        # 如果找到更短的路径
        if current_distance > distances[current_node]:
            continue
        
        # 检查所有邻居
        for neighbor, weight in graph[current_node].items():
            distance = current_distance + weight['weight']
            
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                heapq.heappush(priority_queue, (distance, neighbor))
    
    return float('infinity')

# 创建图并计算最短路径
G = nx.Graph()
G.add_edge('起点A', '节点B', weight=3)
G.add_edge('起点A', '节点C', weight=1)
G.add_edge('节点B', '节点D', weight=2)
G.add_edge('节点C', '节点D', weight=4)
G.add_edge('节点D', '终点E', weight=2)

# 寻找最短路径
shortest_distance = find_shortest_path(G, '起点A', '终点E')

print(f"🔍 图推理示例：最短路径问题")
print(f"\n目标：从起点A到终点E的最短距离")
print(f"\n可能的路径：")
print(f"1. A → B → D → E: 距离 = 3 + 2 + 2 = 7")
print(f"2. A → C → D → E: 距离 = 1 + 4 + 2 = 7")
print(f"3. A → C → B → D → E: 距离 = 1 + 3 + 2 + 2 = 8")
print(f"\n🎯 最短距离: {shortest_distance}")
print(f"\n💡 通过图算法，我们可以高效地找到最优路径！")

## 🎨 实战：思维树vs思维链对比

让我们对比一下Chain-of-Thought和Tree-of-Thought在解决复杂问题时的表现差异。

In [ ]:
def compare_reasoning_methods():
    """比较推理方法"""
    
    # CoT 方法
    cot_result = {
        'method': 'Chain-of-Thought',
        'steps': [
            '第一步：理解题目要求',
            '第二步：计算原价的20%：200×0.2=40元',
            '第三步：计算涨价后的价格：200+40=240元',
            '第四步：计算降价后的价格：240×0.8=192元',
            '第五步：得出最终答案：192元'
        ],
        'answer': 192,
        'time_complexity': 'O(n)',
        'space_complexity': 'O(1)'
    }
    
    # ToT 方法
    tot_result = {
        'method': 'Tree-of-Thought',
        'branches': [
            {
                'name': '分支A: 直接计算',
                'steps': [
                    '200×1.2×0.8 = 192元'
                ],
                'answer': 192
            },
            {
                'name': '分支B: 分步计算',
                'steps': [
                    '涨价：200×1.2=240元',
                    '降价：240×0.8=192元'
                ],
                'answer': 192
            },
            {
                'name': '分支C: 验证计算',
                'steps': [
                    '原价：200元',
                    '变化：+40-48= -8元',
                    '最终：200-8=192元'
                ],
                'answer': 192
            }
        ],
        'final_answer': 192,
        'time_complexity': 'O(b×n)',  # b=branches, n=steps per branch
        'space_complexity': 'O(b)'  # 存储多个分支
    }
    
    # 创建对比图表
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 时间复杂度对比
    methods = ['CoT', 'ToT']
    time_complexity = ['O(n)', 'O(b×n)']
    space_complexity = ['O(1)', 'O(b)']
    
    ax1.bar(methods, [1, 3], color=['skyblue', 'lightcoral'])
    ax1.set_title('时间复杂度对比', fontsize=14, fontweight='bold')
    ax1.set_ylabel('相对计算量')
    for i, v in enumerate(time_complexity):
        ax1.text(i, 0.5, v, ha='center', va='center', fontweight='bold')
    
    # 优势对比雷达图
    categories = ['准确性', '灵活性', '鲁棒性', '可解释性', '效率']
    cot_scores = [8, 6, 7, 9, 9]
    tot_scores = [9, 9, 8, 8, 6]
    
    angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False)
    angles = np.concatenate((angles, [angles[0]]))
    
    cot_scores = np.concatenate((cot_scores, [cot_scores[0]]))
    tot_scores = np.concatenate((tot_scores, [tot_scores[0]]))
    
    ax2 = plt.subplot(1, 2, 2, projection='polar')
    ax2.plot(angles, cot_scores, 'o-', linewidth=2, label='CoT')
    ax2.plot(angles, tot_scores, 'o-', linewidth=2, label='ToT')
    ax2.fill(angles, cot_scores, alpha=0.25)
    ax2.fill(angles, tot_scores, alpha=0.25)
    ax2.set_xticks(angles[:-1])
    ax2.set_xticklabels(categories)
    ax2.set_ylim(0, 10)
    ax2.set_title('推理能力对比 (1-10分)', fontsize=14, fontweight='bold')
    ax2.legend(loc='upper right', bbox_to_anchor=(1.2, 1.0))
    
    plt.tight_layout()
    plt.show()
    
    return cot_result, tot_result

# 执行对比分析
cot_result, tot_result = compare_reasoning_methods()

print("🔄 推理方法对比分析：")
print(f"\n📊 Chain-of-Thought (CoT)：")
print(f"- 答案: {cot_result['answer']}元")
print(f"- 步骤数: {len(cot_result['steps'])}")
print(f"- 时间复杂度: {cot_result['time_complexity']}")
print(f"- 空间复杂度: {cot_result['space_complexity']}")

print(f"\n🌳 Tree-of-Thought (ToT)：")
print(f"- 分支数: {len(tot_result['branches'])}")
print(f"- 最终答案: {tot_result['final_answer']}元")
print(f"- 时间复杂度: {tot_result['time_complexity']}")
print(f"- 空间复杂度: {tot_result['space_complexity']}")
print(f"- 优势: 多路径验证，结果更可靠")

## 🎯 实战练习：思维树应用

让我们实现一个更复杂的思维树应用：**商品价格优化问题**。

### 📋 问题描述
某商店有一批商品，成本价100元，可以定价为80-150元之间。
定价影响销量：价格越高，销量越低。
目标：找到最优价格，使利润最大。

### 🌲 思维树方法
我们需要探索不同定价策略，计算对应的利润。

In [ ]:
def pricing_strategy_analysis():
    """定价策略分析 - 思维树应用"""
    
    # 定义销量函数（价格越高，销量越低）
    def sales_volume(price):
        return max(0, 200 - 2 * price)
    
    # 定义利润函数
    def profit(price, cost=100):
        volume = sales_volume(price)
        return (price - cost) * volume
    
    # 思维树分支：不同定价策略
    strategies = [
        {
            'name': '策略A: 成本加成定价',
            'method': '在成本基础上增加固定比例',
            'price': 120,  # 100 × 1.2
            'reasoning': '成本价100元，加价20%得到120元'
        },
        {
            'name': '策略B: 市场导向定价',
            'method': '参考竞争对手价格',
            'price': 110,  # 竞争对手价格
            'reasoning': '竞争对手定价110元，跟随定价'
        },
        {
            'name': '策略C: 利润最大化',
            'method': '寻找数学最优解',
            'price': None,  # 需要计算
            'reasoning': '通过微积分求导找到利润最大化的价格'
        },
        {
            'name': '策略D: 促销定价',
            'method': '低价快速出货',
            'price': 80,  # 最低价格
            'reasoning': '80元定价快速回笼资金'
        }
    ]
    
    # 计算各策略的利润
    for strategy in strategies:
        if strategy['price'] is not None:
            strategy['profit'] = profit(strategy['price'])
            strategy['volume'] = sales_volume(strategy['price'])
        else:
            # 策略C：数学优化
            # 利润 = (price - 100) * (200 - 2*price)
            # 最优价格 = 150元（数学推导）
            optimal_price = 150
            strategy['price'] = optimal_price
            strategy['profit'] = profit(optimal_price)
            strategy['volume'] = sales_volume(optimal_price)
    
    # 找到最佳策略
    best_strategy = max(strategies, key=lambda x: x['profit'])
    
    # 可视化
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 价格vs利润图
    prices = range(80, 151, 5)
    profits = [profit(p) for p in prices]
    
    ax1.plot(prices, profits, 'b-', linewidth=2, label='利润曲线')
    ax1.axvline(best_strategy['price'], color='red', linestyle='--', 
                label=f'最优价格: {best_strategy["price"]}元')
    ax1.scatter([s['price'] for s in strategies], [s['profit'] for s in strategies], 
                color='orange', s=100, zorder=5, label='各策略点')
    
    ax1.set_xlabel('价格 (元)')
    ax1.set_ylabel('利润 (元)')
    ax1.set_title('价格-利润关系图', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 策略对比表格
    strategy_names = [s['name'] for s in strategies]
    profits = [s['profit'] for s in strategies]
    volumes = [s['volume'] for s in strategies]
    
    x = np.arange(len(strategy_names))
    width = 0.35
    
    ax2.bar(x - width/2, profits, width, label='利润', color='lightblue', alpha=0.8)
    ax2.bar(x + width/2, volumes, width, label='销量', color='lightcoral', alpha=0.8)
    
    ax2.set_xlabel('策略')
    ax2.set_ylabel('数值')
    ax2.set_title('各策略对比', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels([s['name'] for s in strategies], rotation=45, ha='right')
    ax2.legend()
    
    # 标注最佳策略
    best_idx = strategies.index(best_strategy)
    ax2.text(best_idx, best_strategy['profit'] + 50, '最佳', 
             ha='center', va='bottom', fontweight='bold', color='red')
    
    plt.tight_layout()
    plt.show()
    
    return strategies, best_strategy

# 执行定价策略分析
strategies, best_strategy = pricing_strategy_analysis()

print(f"🏪 定价策略分析结果：")
print(f"\n📊 各策略对比：")
for strategy in strategies:
    print(f"\n{strategy['name']}: ")
    print(f"  定价: {strategy['price']}元")
    print(f"  销量: {strategy['volume']}件")
    print(f"  利润: {strategy['profit']}元")
    print(f"  策略: {strategy['method']}")

print(f"\n🎯 最佳策略: {best_strategy['name']}")
print(f"最优定价: {best_strategy['price']}元")
print(f"最大利润: {best_strategy['profit']}元")
print(f"预期销量: {best_strategy['volume']}件")

## 🧩 练习题与思考

### 💡 实践题1：实现一个简单的思维树求解器

请实现一个可以解决数学问题的思维树求解器，支持多种解题方法。

### 💡 实践题2：图路径优化

给定一个复杂的交通网络图，找到从A到B的最短路径。

### 💡 思考题

1. 在什么情况下Tree-of-Thought比Chain-of-Thought更有效？
2. 思维树的剪枝策略有哪些？如何实现？
3. 如何评估思维树推理的质量？

## 📚 总结

今天我们学习了Tree-of-Thought和图推理的核心概念：

### ✅ 核心要点
- **Tree-of-Thought**：多分支并行推理
- **图推理**：基于图结构的最优路径搜索
- **自洽性检查**：确保多路径逻辑一致
- **剪枝策略**：优化搜索效率
- **实际应用**：价格优化、路径规划、决策制定

### 🌟 下一步
- 尝试实现更复杂的思维树应用
- 学习图神经网络的结合
- 探索深度思维树（DeepToT）

### 🎯 学习检测
- 能否解释Tree-of-Thought相比CoT的优势？
- 能否实现一个简单的思维树求解器？
- 理解图推理中的剪枝和自洽性概念？

### 🔗 延伸学习
- 深入阅读Tree-of-Thought论文
- 学习A*算法在图推理中的应用
- 探索思维链与思维树的混合方法